In [1]:
!pip install torch_optimizer
!pip install medpy
!pip install albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=medpy-0.5.2-py3-none-any.whl size=224805 sha256=0a1bfe875580a2bf1b696b1a46205c4630042c9c084ca78ac122d7f2d0b265cf
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
from PIL import Image

from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.patches as mpatches

In [3]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF
import random


class SynapseDataset(Dataset):
    def __init__(self, npz_files, augment=False, image_size=(112, 112)):
        self.files = []

        for f in npz_files:
            npz = np.load(f)
            image, label = npz['image'], npz['label']

            if np.max(image) > 0 and np.max(label) > 0:
                self.files.append(f)

        self.augment = augment
        self.image_size = image_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        npz = np.load(self.files[idx])

        image = npz['image'].astype(np.float32)
        label = npz['label'].astype(np.int64)

        # Min-max normalization
        image = (image - image.min()) / (
            image.max() - image.min() + 1e-8
        )

        image = np.expand_dims(image, axis=0)

        image = torch.from_numpy(image)
        label = torch.from_numpy(label)

        # Resize
        image = TF.resize(image, self.image_size)

        label = TF.resize(
            label.unsqueeze(0).float(),
            self.image_size,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0).long()

        # Slightly stronger augmentation
        if self.augment:
            image, label = self.random_augment(image, label)

        return image, label

    def random_augment(self, image, label):

        # Horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label)

        # Slightly larger rotation
        angle = random.uniform(-10, 10)

        image = TF.rotate(
            image,
            angle,
            interpolation=TF.InterpolationMode.BILINEAR
        )

        label = TF.rotate(
            label.unsqueeze(0),
            angle,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0)

        # Slight brightness variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_brightness(image, factor)

        # Slight contrast variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_contrast(image, factor)

        return image, label

In [4]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

dataset_path = "/kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz"
all_npz_files = sorted([os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.npz')])

# Split
train_files, val_files = train_test_split(all_npz_files, test_size=0.25, random_state=42)

train_dataset = SynapseDataset(train_files)
val_dataset = SynapseDataset(val_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4)

In [5]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape) 
print("Mask batch shape:", masks.shape)   

Image batch shape: torch.Size([4, 1, 112, 112])
Mask batch shape: torch.Size([4, 112, 112])


In [6]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.spatial_att = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=kernel_size, stride=1, padding=(kernel_size-1)//2),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.channel_att(x)
        x = x * ca
        sa = torch.cat([x.max(1, keepdim=True)[0], x.mean(1, keepdim=True)], dim=1)
        sa = self.spatial_att(sa)
        x = x * sa
        return x

class ViTBlock(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(self, dim=768, num_layers=12):
        super().__init__()
        self.layers = nn.ModuleList([ViTBlock(dim) for _ in range(num_layers)])

    def forward(self, x):
        B, C, H, W = x.shape
        seq = x.flatten(2).transpose(1, 2)  
        feats = []
        for i, blk in enumerate(self.layers):
            seq = blk(seq)
            if i in [2, 5, 8, 11]:
                tmp = seq.transpose(1, 2).reshape(B, C, H, W)
                feats.append(tmp)
        return feats  

class HybridSkipAttention(nn.Module):
    def __init__(self, res_ch, vit_ch, out_ch, res_up=1, vit_up=1, num_heads=8):
        super().__init__()
        self.res_conv = nn.Conv2d(res_ch, out_ch, 3, padding=1)
        self.vit_conv = nn.Conv2d(vit_ch, out_ch, 3, padding=1)

        self.res_deconvs = nn.ModuleList([nn.ConvTranspose2d(out_ch, out_ch, 2, 2) for _ in range(res_up)])
        self.vit_deconvs = nn.ModuleList([nn.ConvTranspose2d(out_ch, out_ch, 2, 2) for _ in range(vit_up)])

        self.fuse_conv = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.cbam = CBAM(out_ch)

        self.norm1 = nn.LayerNorm(out_ch)
        self.attn  = nn.MultiheadAttention(out_ch, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(out_ch)
        self.mlp   = nn.Sequential(
            nn.Linear(out_ch, out_ch*4),
            nn.GELU(),
            nn.Linear(out_ch*4, out_ch)
        )

    def forward(self, res_feat, vit_feat, return_att=False):
        r = self.res_conv(res_feat)
        v = self.vit_conv(vit_feat)

        for de in self.res_deconvs: r = de(r)
        for de in self.vit_deconvs: v = de(v)

        min_h = min(r.shape[2], v.shape[2])
        min_w = min(r.shape[3], v.shape[3])
        r = r[:, :, :min_h, :min_w]
        v = v[:, :, :min_h, :min_w]

        x = r + v
        x = self.fuse_conv(x)

        B, C, H, W = x.shape
        seq = x.flatten(2).transpose(1, 2)
        seq_attn, attn_weights = self.attn(self.norm1(seq), self.norm1(seq), self.norm1(seq))
        seq = seq + seq_attn
        seq = seq + self.mlp(self.norm2(seq))
        x = seq.transpose(1, 2).reshape(B, C, H, W)

        x = self.cbam(x)

        if return_att:
            return x, attn_weights
        else:
            return x

class BottleneckFusion(nn.Module):
    """Cross-attention removed: plain concat + conv fusion (normal UNet-style bottleneck)."""
    def __init__(self, res_in_ch=2048, vit_in_ch=768, out_ch=768):
        super().__init__()
        self.res_up = nn.ConvTranspose2d(res_in_ch, res_in_ch, kernel_size=2, stride=2)
        self.conv = nn.Conv2d(res_in_ch + vit_in_ch, out_ch, 3, padding=1)

    def forward(self, res4, vit12):
        r = self.res_up(res4)

        min_h = min(r.shape[2], vit12.shape[2])
        min_w = min(r.shape[3], vit12.shape[3])
        r = r[:, :, :min_h, :min_w]
        vit12 = vit12[:, :, :min_h, :min_w]

        x = torch.cat([r, vit12], dim=1)
        x = self.conv(x)
        return x

class SegmentationModel(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)

        old_conv1 = backbone.conv1
        backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv1.out_channels,
            kernel_size=old_conv1.kernel_size,
            stride=old_conv1.stride,
            padding=old_conv1.padding,
            bias=False
        )
        with torch.no_grad():
            backbone.conv1.weight[:] = old_conv1.weight.mean(dim=1, keepdim=True)

        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # 256 ch
        self.layer2 = backbone.layer2   # 512 ch
        self.layer3 = backbone.layer3   # 1024 ch
        self.layer4 = backbone.layer4   # 2048 ch

        self.l3_to_vit = nn.Sequential(
            nn.Conv2d(1024, 768, 3, padding=1),
            nn.Conv2d(768, 768, 3, padding=1)
        )

        self.vit = ViTEncoder(dim=768, num_layers=12)
        self.bottleneck = BottleneckFusion(res_in_ch=2048, vit_in_ch=768, out_ch=768)

        # Decoder
        self.up1 = nn.ConvTranspose2d(768, 512, 2, 2)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.up4 = nn.ConvTranspose2d(128,  64, 2, 2)

        # Hybrid skip fusions
        self.skip1 = HybridSkipAttention(res_ch=1024, vit_ch=768, out_ch=512, res_up=1, vit_up=1)
        self.skip2 = HybridSkipAttention(res_ch=512,  vit_ch=768, out_ch=256, res_up=1, vit_up=2)
        self.skip3 = HybridSkipAttention(res_ch=256,  vit_ch=768, out_ch=128, res_up=1, vit_up=3)

        # Input skip 
        self.input_skip = nn.Conv2d(1, 64, 3, padding=1)

        self.final_conv = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        l0 = self.stem(x)      
        l1 = self.layer1(l0)   
        l2 = self.layer2(l1)   
        l3 = self.layer3(l2)     
        l4 = self.layer4(l3)   

        vit_in = self.l3_to_vit(l3)                      
        vit_L3, vit_L6, vit_L9, vit_L12 = self.vit(vit_in)

        b = self.bottleneck(l4, vit_L12)                

        up1 = self.up1(b)   + self.skip1(l3, vit_L9)     
        up2 = self.up2(up1) + self.skip2(l2, vit_L6)      
        up3 = self.up3(up2) + self.skip3(l1, vit_L3)     
        up4 = self.up4(up3) + self.input_skip(x)          

        out = self.final_conv(up4)                        
        return out

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SegmentationModel(num_classes=9).to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s] 


In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from medpy.metric.binary import hd

def dice_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = preds_onehot.sum(dim=(0,2,3)) + targets_onehot.sum(dim=(0,2,3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean().item()

def iou_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = (preds_onehot + targets_onehot - preds_onehot*targets_onehot).sum(dim=(0,2,3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def pixel_accuracy(preds, targets):
    return (preds.argmax(dim=1) == targets).float().mean().item()

def precision_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fp = (preds_onehot * (1 - targets_onehot)).sum(dim=(0,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    return precision.mean().item()

def recall_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fn = ((1 - preds_onehot) * targets_onehot).sum(dim=(0,2,3))
    recall = (tp + eps) / (tp + fn + eps)
    return recall.mean().item()

def f1_score(preds, targets, eps=1e-7):
    prec = precision_score(preds, targets, eps)
    rec = recall_score(preds, targets, eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return f1

def hausdorff_distance(preds, targets):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    distances = []
    for b in range(preds_onehot.shape[0]):
        for c in range(preds_onehot.shape[1]):
            p = preds_onehot[b,c].cpu().numpy().astype(bool)
            t = targets_onehot[b,c].cpu().numpy().astype(bool)
            if p.sum() == 0 or t.sum() == 0:
                distances.append(np.nan)
            else:
                try:
                    distances.append(hd(p, t))
                except:
                    distances.append(np.nan)
    return np.nanmean(distances)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, preds, targets):
        num_classes = preds.shape[1]
        preds = F.softmax(preds, dim=1)
        preds_one_hot = F.one_hot(preds.argmax(dim=1), num_classes=num_classes).permute(0,3,1,2).float()
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0,3,1,2).float()

        intersection = (preds_one_hot * targets_one_hot).sum(dim=(0,2,3))
        union = preds_one_hot.sum(dim=(0,2,3)) + targets_one_hot.sum(dim=(0,2,3))
        dice = (2. * intersection + self.eps) / (union + self.eps)
        return 1 - dice.mean()

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def combined_loss(outputs, targets, ce_weight=0.5, dice_weight=0.5):
    return ce_weight * ce_loss(outputs, targets) + dice_weight * dice_loss(outputs, targets)

criterion = combined_loss
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=4, factor=0.5)

In [10]:
from torch.amp import autocast, GradScaler
import torch
import torch.nn.functional as F
from tqdm import tqdm

scaler = GradScaler() 
accumulation_steps = 4
num_epochs =100

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_dice = train_acc = train_iou = train_hd = 0.0
    train_mean_iou = train_precision = train_recall = train_f1 = 0.0

    optimizer.zero_grad()

    for step, (images, masks) in tqdm(enumerate(train_loader)):
        images, masks = images.to(device), masks.to(device)

        with autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)  # [B, C, H, W]
            loss = criterion(outputs, masks) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps  
        train_dice += dice_score(outputs, masks)
        train_acc += pixel_accuracy(outputs, masks)
        train_iou += iou_score(outputs, masks)
        train_hd += hausdorff_distance(outputs, masks)
        train_mean_iou += iou_score(outputs, masks)  
        train_precision += precision_score(outputs, masks)
        train_recall += recall_score(outputs, masks)
        train_f1 += f1_score(outputs, masks)

    n_train = len(train_loader)
    avg_train_loss = train_loss / n_train
    avg_train_dice = train_dice / n_train
    avg_train_acc = train_acc / n_train
    avg_train_iou = train_iou / n_train
    avg_train_hd = train_hd / n_train
    avg_train_mean_iou = train_mean_iou / n_train
    avg_train_precision = train_precision / n_train
    avg_train_recall = train_recall / n_train
    avg_train_f1 = train_f1 / n_train

    model.eval()
    val_loss = 0.0
    val_dice = val_acc = val_iou = val_hd = 0.0
    val_mean_iou = val_precision = val_recall = val_f1 = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader):
            images, masks = images.to(device), masks.to(device)
            with autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_score(outputs, masks)
            val_acc += pixel_accuracy(outputs, masks)
            val_iou += iou_score(outputs, masks)
            val_hd += hausdorff_distance(outputs, masks)
            val_mean_iou += iou_score(outputs, masks)
            val_precision += precision_score(outputs, masks)
            val_recall += recall_score(outputs, masks)
            val_f1 += f1_score(outputs, masks)

    n_val = len(val_loader)
    avg_val_loss = val_loss / n_val
    avg_val_dice = val_dice / n_val
    avg_val_acc = val_acc / n_val
    avg_val_iou = val_iou / n_val
    avg_val_hd = val_hd / n_val
    avg_val_mean_iou = val_mean_iou / n_val
    avg_val_precision = val_precision / n_val
    avg_val_recall = val_recall / n_val
    avg_val_f1 = val_f1 / n_val

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch:2d} | "
          f"Train Loss: {avg_train_loss:.6f} | Dice: {avg_train_dice:.4f} | Acc: {avg_train_acc:.4f} | "
          f"IoU: {avg_train_iou:.4f} | mIoU: {avg_train_mean_iou:.4f} | Precision: {avg_train_precision:.4f} | Recall: {avg_train_recall:.4f} | F1: {avg_train_f1:.4f} | HD: {avg_train_hd:.4f} || "
          f"Val Loss: {avg_val_loss:.6f} | Dice: {avg_val_dice:.4f} | Acc: {avg_val_acc:.4f} | "
          f"IoU: {avg_val_iou:.4f} | mIoU: {avg_val_mean_iou:.4f} | Precision: {avg_val_precision:.4f} | Recall: {avg_val_recall:.4f} | F1: {avg_val_f1:.4f} | HD: {avg_val_hd:.4f}")

240it [01:03,  3.79it/s]
100%|██████████| 81/81 [00:09<00:00,  8.46it/s]

Epoch  1 | Train Loss: 0.781766 | Dice: 0.1927 | Acc: 0.8177 | IoU: 0.1874 | mIoU: 0.1874 | Precision: 0.8447 | Recall: 0.2282 | F1: 0.3229 | HD: 54.3132 || Val Loss: 0.575824 | Dice: 0.2083 | Acc: 0.9252 | IoU: 0.2043 | mIoU: 0.2043 | Precision: 0.9917 | Recall: 0.2126 | F1: 0.3319 | HD: 51.4383



240it [00:58,  4.11it/s]
100%|██████████| 81/81 [00:09<00:00,  8.43it/s]

Epoch  2 | Train Loss: 0.531320 | Dice: 0.2143 | Acc: 0.9264 | IoU: 0.2063 | mIoU: 0.2063 | Precision: 0.8927 | Recall: 0.2278 | F1: 0.3423 | HD: 41.4389 || Val Loss: 0.501909 | Dice: 0.2292 | Acc: 0.9304 | IoU: 0.2169 | mIoU: 0.2169 | Precision: 0.9626 | Recall: 0.2249 | F1: 0.3476 | HD: 35.8375



240it [00:59,  4.05it/s]
100%|██████████| 81/81 [00:10<00:00,  7.79it/s]

...

Epoch 98 | Train Loss: 0.043895 | Dice: 0.9257 | Acc: 0.9947 | IoU: 0.8800 | mIoU: 0.8800 | Precision: 0.9456 | Recall: 0.9187 | F1: 0.9315 | HD: 2.0368 || Val Loss: 0.084676 | Dice: 0.8566 | Acc: 0.9903 | IoU: 0.7903 | mIoU: 0.7903 | Precision: 0.9084 | Recall: 0.8481 | F1: 0.8761 | HD: 3.6719



240it [01:02,  3.81it/s]
100%|██████████| 81/81 [00:11<00:00,  7.27it/s]

Epoch 99 | Train Loss: 0.043246 | Dice: 0.9270 | Acc: 0.9947 | IoU: 0.8821 | mIoU: 0.8821 | Precision: 0.9474 | Recall: 0.9205 | F1: 0.9333 | HD: 2.0577 || Val Loss: 0.085960 | Dice: 0.8539 | Acc: 0.9904 | IoU: 0.7878 | mIoU: 0.7878 | Precision: 0.9069 | Recall: 0.8447 | F1: 0.8737 | HD: 3.6308



240it [01:02,  3.84it/s]
100%|██████████| 81/81 [00:10<00:00,  7.39it/s]

Epoch 100 | Train Loss: 0.043527 | Dice: 0.9264 | Acc: 0.9948 | IoU: 0.8817 | mIoU: 0.8817 | Precision: 0.9479 | Recall: 0.9205 | F1: 0.9335 | HD: 2.0373 || Val Loss: 0.084020 | Dice: 0.8579 | Acc: 0.9904 | IoU: 0.7918 | mIoU: 0.7918 | Precision: 0.9048 | Recall: 0.8499 | F1: 0.8755 | HD: 3.6699
